# Train Seq2SeqSimplifier trên Colab

Notebook này chỉ lo phần môi trường (clone code, gắn dữ liệu/model từ Drive, cài thư viện) —
logic train thật sự nằm trong repo (`src/simplification/seq2seq_simplifier.py`,
`scripts/simplification/train_seq2seq_simplifier.py`), đồng bộ qua git.

**Trước khi chạy:**
1. Đổi `DRIVE_ROOT` bên dưới nếu bạn để dữ liệu ở thư mục Drive khác.
2. Đã upload sẵn `data/dataset/simplificated_spo_sentence.csv` vào `DRIVE_ROOT/data/dataset/simplificated_spo_sentence.csv` trên Drive.
3. Chọn Runtime > Change runtime type > GPU trước khi chạy (nếu có GPU free trên Colab).


## 1. Mount Google Drive

In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 2. Clone / pull code từ git

Repo public, không cần token. Nếu đã clone từ lần trước, cell này sẽ `git pull` thay vì clone lại.

In [2]:
REPO_URL = "https://github.com/thnghia-ctu/CausalGraph.git"
BRANCH = "v3"
REPO_DIR = "/content/CausalGraph"

import os

if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
    !git checkout $BRANCH
    !git pull origin $BRANCH
else:
    !git clone -b $BRANCH $REPO_URL $REPO_DIR
    %cd $REPO_DIR


Cloning into '/content/CausalGraph'...
remote: Enumerating objects: 833, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 833 (delta 42), reused 74 (delta 27), pack-reused 718 (from 1)
Receiving objects: 100% (833/833), 1.82 MiB | 4.70 MiB/s, done.
Resolving deltas: 100% (425/425), done.
/content/CausalGraph


## 3. Gắn `data/` và `models/` vào Drive

Hai thư mục này bị `.gitignore`, không nằm trong git — clone xong sẽ trống hoặc không tồn tại.
Symlink sang Drive để dữ liệu và checkpoint được giữ lại qua các session, không cần copy tay
mỗi lần mở lại Colab.

In [3]:
DRIVE_ROOT = "/content/drive/MyDrive/CausalGraph"

import os

os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/models", exist_ok=True)

!rm -rf {REPO_DIR}/data {REPO_DIR}/models
!ln -s {DRIVE_ROOT}/data {REPO_DIR}/data
!ln -s {DRIVE_ROOT}/models {REPO_DIR}/models

!ls -la {REPO_DIR}/data/dataset/ 2>/dev/null || echo "Chưa có data/dataset/simplificated_spo_sentence.csv trên Drive — upload trước khi train."


total 6419
-rw------- 1 root root 2926373 Aug  9 15:15 causal_sentences.csv
-rw------- 1 root root 3646112 Aug 10 01:34 simplificated_spo_sentence.csv


## 4. Cài thư viện

In [4]:
!pip install -q -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 18.2 MB/s eta 0:00:00
   ━━━━

## 5. Đăng nhập Hugging Face Hub

Bắt buộc — cell train ở bước 6 giờ tự push model lên Hub ngay khi train xong
(dùng repo ID khai trong `configs/config.py`). Token tạo tại
https://huggingface.co/settings/tokens (quyền write).


In [5]:
from huggingface_hub import notebook_login

notebook_login()


## 6. Train

Checkpoint được lưu định kỳ vào `models/simplifier/` (= Drive, qua symlink ở bước 3).
Nếu Colab bị ngắt kết nối giữa chừng, chỉ cần chạy lại cell này — `Seq2SeqSimplifier.fit`
tự resume từ checkpoint gần nhất thay vì train lại từ đầu. Train xong sẽ tự
push model lên Hugging Face Hub (repo ID lấy từ configs/config.py).

In [6]:
!python -m scripts.simplification.train_seq2seq_simplifier


config.json: 100% 702/702 [00:00<00:00, 972kB/s]
tokenizer_config.json: 100% 2.20k/2.20k [00:00<00:00, 1.04MB/s]

spiece.model: downloading bytes:   0% 0.00/820k [00:00<?, ?B/s]
spiece.model: downloading bytes: 100% 615k/615k [00:00<00:00, 865kB/s, 60.9kB/s  ]
spiece.model: reconstructing file: 100% 820k/820k [00:00<00:00, 1.15MB/s, 81.2kB/s  ]
tokenizer.json: 100% 2.40M/2.40M [00:00<00:00, 19.7MB/s]
special_tokens_map.json: 100% 2.12k/2.12k [00:00<00:00, 8.55MB/s]

pytorch_model.bin: downloading bytes:   3% 23.6M/904M [00:01<00:26, 32.6MB/s, 1.20MB/s  ]
pytorch_model.bin: downloading bytes:  17% 155M/904M [00:01<00:05, 131MB/s, 13.1MB/s  ]
pytorch_model.bin: downloading bytes:  43% 390M/904M [00:03<00:02, 202MB/s, 33.0MB/s  ]
pytorch_model.bin: reconstructing file:  52% 469M/904M [00:03<00:02, 161MB/s, 25.1MB/s  ]
pytorch_model.bin: downloading bytes: 100% 401M/401M [00:06<00:00, 57.5MB/s, 34.4MB/s  ]
pytorch_model.bin: reconstructing file: 100% 904M/904M [00:06<00:00, 130MB/s, 66.7MB